# AI Automation in End-to-End Data Workflows (Databricks)

**Auto-tagging · Report generation · Chatbots — all with built-in AI Functions.**

This notebook is a runnable companion to the guide. It uses small synthetic datasets so you can run every cell top-to-bottom and see real model output, then swap in your own tables.

### What you'll build
1. **Auto-tagging** — classify, score sentiment, and extract entities from support tickets in one query.
2. **Report generation** — turn Gold-layer metrics into a written executive summary.
3. **Chatbot (RAG)** — answer questions grounded on your own knowledge base.
4. **End-to-end** — wire the pieces into a medallion-style flow you can schedule.

---
### ⚙️ Requirements (read before running)
- **Serverless compute** (notebook attached to Serverless) **or a Serverless SQL warehouse**. The AI functions here are *not* available on Classic/Pro warehouses or classic clusters.
- **Databricks Runtime 18.2+** for `ai_query`.
- Your workspace must be in a **Model Serving–supported region** with Foundation Model APIs enabled.
- No API keys needed — Databricks-hosted models are called by name.


## 0 · Setup & synthetic data

We create temp views so the notebook needs **no catalog write permissions**. In production you'd persist these as Delta tables (Bronze/Silver/Gold) — see the last section.

In [0]:
# Config — point these at your own catalog.schema when you productionise
CHAT_MODEL = 'databricks-llama-4-maverick'  # any Foundation Model / served endpoint name

print('Spark:', spark.version)
print('Chat model endpoint:', CHAT_MODEL)

Spark: 4.1.0
Chat model endpoint: databricks-llama-4-maverick


In [0]:
# --- Synthetic support tickets (unstructured text) ---
tickets = [
    (1, 'I was charged twice for order #A-4471 this month. Please refund the duplicate.'),
    (2, 'The mobile app crashes every time I open the reports tab on Android 14.'),
    (3, 'Love the product! Could you add a dark mode option in settings?'),
    (4, 'I cannot log in — the password reset email for jane@acme.co never arrives.'),
    (5, 'Billing invoice INV-2231 shows the wrong tax rate for my region.'),
    (6, 'Export to CSV has been broken since the last update, super frustrating.'),
]
spark.createDataFrame(tickets, 'ticket_id INT, body STRING').createOrReplaceTempView('tickets')
display(spark.table('tickets'))

ticket_id,body
1,I was charged twice for order #A-4471 this month. Please refund the duplicate.
2,The mobile app crashes every time I open the reports tab on Android 14.
3,Love the product! Could you add a dark mode option in settings?
4,I cannot log in — the password reset email for jane@acme.co never arrives.
5,Billing invoice INV-2231 shows the wrong tax rate for my region.
6,"Export to CSV has been broken since the last update, super frustrating."


## 1 · Auto-tagging

One query classifies each ticket, scores its sentiment, and extracts structured fields. These become normal columns you can filter, join, and route on.

- `ai_classify(text, labels, options)` → one of your labels (v2.0 recommended)
- `ai_analyze_sentiment(text)` → positive / negative / mixed / neutral
- `ai_extract(text, fields)` → a struct of the fields you name

In [0]:
tagged = spark.sql("""
    SELECT
        ticket_id,
        body,
        ai_classify(
            body,
            '["Billing", "Bug", "Feature Request", "Account Access"]',
            map('version', '2.0')
        ):response[0]::string                              AS category,
        ai_analyze_sentiment(body)                         AS sentiment,
        ai_extract(body, ARRAY('order_number', 'email'))   AS entities
    FROM tickets
""")
tagged.createOrReplaceTempView('tickets_tagged')
display(tagged)

ticket_id,body,category,sentiment,entities
1,I was charged twice for order #A-4471 this month. Please refund the duplicate.,Billing,negative,"List(A-4471, null)"
2,The mobile app crashes every time I open the reports tab on Android 14.,Bug,negative,"List(null, null)"
3,Love the product! Could you add a dark mode option in settings?,Feature Request,mixed,"List(null, null)"
4,I cannot log in — the password reset email for jane@acme.co never arrives.,Account Access,negative,"List(null, jane@acme.co)"
5,Billing invoice INV-2231 shows the wrong tax rate for my region.,Billing,negative,"List(INV-2231, null)"
6,"Export to CSV has been broken since the last update, super frustrating.",Bug,negative,"List(null, null)"


**Route on the tags.** Because the AI output is just columns, downstream logic is plain SQL — no model call needed here.

In [0]:
display(spark.sql("""
    SELECT category,
           count(*)                                              AS volume,
           sum(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END) AS negative_cnt
    FROM tickets_tagged
    GROUP BY category
    ORDER BY volume DESC
"""))

category,volume,negative_cnt
Billing,2,2
Bug,2,2
Feature Request,1,0
Account Access,1,1


## 2 · Automated report generation

Two steps, always: **(a)** compute the numbers in SQL, **(b)** let the model narrate them. The model never does arithmetic — it only phrases the figures you give it, which keeps reports auditable.

First, some Gold-layer sales metrics:

In [0]:
sales = [
    ('North',  120, 48200.0), ('South',  95, 39900.0),
    ('East',   150, 61250.0), ('West',   60, 21750.0),
]
spark.createDataFrame(sales, 'region STRING, orders INT, revenue DOUBLE') \
     .createOrReplaceTempView('weekly_sales')
display(spark.table('weekly_sales'))

region,orders,revenue
North,120,48200.0
South,95,39900.0
East,150,61250.0
West,60,21750.0


### 2a · Narrative executive summary with `ai_gen`

In [0]:
report = spark.sql("""
    WITH kpis AS (
      SELECT concat(
        'Total revenue $', round(sum(revenue)/1000, 1), 'K across ',
        sum(orders), ' orders. Top region: East. Weakest: West. ',
        'Avg order value $', round(sum(revenue)/sum(orders), 2), '.'
      ) AS facts
      FROM weekly_sales
    )
    SELECT
      facts,
      ai_gen(
        concat(
          'Write a concise 3-sentence weekly sales summary for a non-technical ',
          'VP. Be factual, use only these figures, no fluff: ', facts
        )
      ) AS exec_summary
    FROM kpis
""")
row = report.first()
print('FACTS   :', row['facts'])
print()
print('SUMMARY :', row['exec_summary'])

FACTS   : Total revenue $171.1K across 425 orders. Top region: East. Weakest: West. Avg order value $402.59.

SUMMARY : Our total revenue for the week was $171,100, generated from 425 orders. The East region performed the strongest, while the West region was the weakest in terms of sales. The average order value for the week was $402.59, indicating a consistent level of customer spending across our 425 orders.


### 2b · Per-segment commentary + summarising long text

`ai_gen` can run per row for segment-level commentary, and `ai_summarize(text, max_words)` condenses long free text such as call transcripts.

In [0]:
display(spark.sql("""
    SELECT region, orders, revenue,
           ai_gen(concat(
               'In one short sentence, comment on this region''s weekly performance. ',
               'Region ', region, ': ', orders, ' orders, $', revenue, ' revenue.'
           )) AS commentary
    FROM weekly_sales
"""))

region,orders,revenue,commentary
North,120,48200.0,"Region North had a strong weekly performance, generating $48,200 in revenue from 120 orders."
South,95,39900.0,"Region South had a strong weekly performance, generating nearly $40,000 in revenue from 95 orders."
East,150,61250.0,"Region East had a strong weekly performance, generating $61,250 in revenue from 150 orders."
West,60,21750.0,"Region West had a strong weekly performance, generating $21,750 in revenue from 60 orders."


## 3 · Chatbot (RAG) grounded on your data

A grounded chatbot = **retrieve** the most relevant knowledge, then **generate** an answer from only that context.

For a small knowledge base we rank chunks with `ai_similarity` (semantic score) and skip a vector index. Past a few hundred docs, sync a **Mosaic AI Vector Search** index off a Delta table and retrieve from it instead — the generate step stays identical.

In [0]:
# A tiny company knowledge base (would be chunked docs in real life)
kb = [
    ('kb1', 'Refunds are processed within 5 to 7 business days to the original payment method.'),
    ('kb2', 'You can enable dark mode under Settings > Appearance. It is available on web and mobile.'),
    ('kb3', 'Password reset links expire after 30 minutes. Check spam if the email does not arrive.'),
    ('kb4', 'CSV export supports up to 1 million rows. Larger exports should use the API.'),
    ('kb5', 'Invoices can be downloaded from Billing > History. Tax rate is set by your account region.'),
]
spark.createDataFrame(kb, 'doc_id STRING, chunk STRING').createOrReplaceTempView('kb')
display(spark.table('kb'))

doc_id,chunk
kb1,Refunds are processed within 5 to 7 business days to the original payment method.
kb2,You can enable dark mode under Settings > Appearance. It is available on web and mobile.
kb3,Password reset links expire after 30 minutes. Check spam if the email does not arrive.
kb4,CSV export supports up to 1 million rows. Larger exports should use the API.
kb5,Invoices can be downloaded from Billing > History. Tax rate is set by your account region.


In [0]:
CHAT_MODEL = 'databricks-llama-4-maverick'
CHAT_MODEL = globals().get('CHAT_MODEL', 'databricks-llama-4-maverick')

In [0]:
def ask(question: str, top_k: int = 2) -> str:
    # 1) RETRIEVE: rank KB chunks by semantic similarity to the question
    ctx = spark.sql(f"""
        SELECT chunk, ai_similarity(chunk, '{question.replace("'", "''")}') AS score
        FROM kb ORDER BY score DESC LIMIT {top_k}
    """)
    context = '\n'.join(r['chunk'] for r in ctx.collect())

    # 2) GENERATE: answer using ONLY the retrieved context
    prompt = (
        'Answer the question using ONLY the context below. '
        'If the answer is not in the context, say you do not know.\n\n'
        f'Context:\n{context}\n\nQuestion: {question}'
    )
    ans = spark.sql(
        'SELECT ai_query(:m, :p) AS a',
        args={'m': CHAT_MODEL, 'p': prompt}
    ).first()['a']
    return ans

print(ask('How long do refunds take?'))
print('---')
print(ask('How do I turn on dark mode?'))
print('---')
print(ask('What is your office address?'))  # not in KB -> should decline

Refunds are processed within 5 to 7 business days to the original payment method.
---
You can enable dark mode under Settings > Appearance.
---
I do not know.


## 4 · End-to-end: the governed pipeline

The three patterns are stages of one flow on the medallion architecture. Here we simulate it in-notebook; in production each stage is a Delta table and a scheduled task.

| Stage | Layer | AI role |
|---|---|---|
| Ingest raw tickets | Bronze | none |
| Clean + auto-tag | Silver | Pattern 1 |
| Aggregate metrics | Gold | none |
| Narrate report | Gold+ | Pattern 2 |
| Serve chatbot | App | Pattern 3 |

In [0]:
# SILVER: enrich once, persist the tags (here: a view; in prod write to Delta)
spark.sql("""
    CREATE OR REPLACE TEMP VIEW silver_tickets AS
    SELECT ticket_id, body,
           ai_classify(body,
                       '["Billing","Bug","Feature Request","Account Access"]',
                       map('version','2.0')):response[0]::string  AS category,
           ai_analyze_sentiment(body)                             AS sentiment
    FROM tickets
""")

# GOLD: aggregate (plain SQL, no AI)
spark.sql("""
    CREATE OR REPLACE TEMP VIEW gold_ticket_stats AS
    SELECT category, count(*) AS volume,
           round(avg(CASE WHEN sentiment='negative' THEN 1.0 ELSE 0.0 END),2) AS neg_rate
    FROM silver_tickets GROUP BY category
""")
display(spark.table('gold_ticket_stats'))

category,volume,neg_rate
Billing,2,1.00
Bug,2,1.00
Feature Request,1,0.00
Account Access,1,1.00


In [0]:
# GOLD+: narrate the Gold stats into an ops briefing
facts = ' | '.join(
    f"{r['category']}: {r['volume']} tickets, {int(r['neg_rate']*100)}% negative"
    for r in spark.table('gold_ticket_stats').collect()
)
briefing = spark.sql(
    'SELECT ai_gen(:p) AS b',
    args={'p': 'Write a 2-sentence support ops briefing highlighting the biggest risk area. Data: ' + facts}
).first()['b']
print(briefing)

Our current support operations are facing a significant risk in the areas of Billing and Bug resolution, with 100% of tickets in these categories receiving negative feedback, indicating a high level of customer dissatisfaction. The high negative sentiment in these areas, combined with the fact that they account for 4 out of 5 total tickets, suggests that addressing these issues should be our top priority to improve overall customer experience and reduce the risk of further escalation.


### Scheduling & productionising
- Replace temp views with **Delta tables** (`CREATE TABLE ... USING DELTA` or Auto Loader for Bronze).
- Put the Silver-enrich and Gold-narrate cells in **notebook tasks** in a **Databricks Workflow** (or **Lakeflow Declarative Pipelines / DLT**) on a schedule.
- Deploy the `ask()` chatbot behind a **Model Serving endpoint** or a Databricks App; for multi-step/tool use, move to the **Mosaic AI Agent Framework**.
- **Govern & control cost:** Unity Catalog on every table/endpoint · `ai_mask()` PII before prompting · enrich once into Silver (don't re-infer per query) · track spend via the AI Gateway usage tables.

---
*Function names and syntax reflect Databricks docs current as of July 2026. Confirm against your workspace before production use.*